In [3]:

!pip -q install datasets transformers evaluate scikit-learn torch accelerate matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00


In [4]:
import os
os.environ["DISABLE_MLFLOW_INTEGRATION"] = "TRUE"
os.environ["WANDB_DISABLED"] = "true"

import random
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [5]:
from google.colab import output
output.disable_custom_widget_manager()

In [6]:
from datasets import load_dataset

dataset = load_dataset("glue", "qqp")

print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

qqp/train-00000-of-00001.parquet:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

qqp/validation-00000-of-00001.parquet:   0%|          | 0.00/3.73M [00:00<?, ?B/s]

qqp/test-00000-of-00001.parquet:   0%|          | 0.00/36.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 390965
    })
})
{'question1': 'How is the life of a math student? Could you describe your own experiences?', 'question2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0}


In [7]:
train_df = pd.DataFrame(dataset["train"])
val_df = pd.DataFrame(dataset["validation"])

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("\nTrain label distribution:")
print(train_df["label"].value_counts(normalize=True).rename("proportion"))
print(train_df["label"].value_counts().rename("count"))

train_df["q1_len"] = train_df["question1"].fillna("").str.split().str.len()
train_df["q2_len"] = train_df["question2"].fillna("").str.split().str.len()
print("\nQuestion length summary:")
print(train_df[["q1_len", "q2_len"]].describe())

Train size: 363846
Validation size: 40430

Train label distribution:
label
0    0.630673
1    0.369327
Name: proportion, dtype: float64
label
0    229468
1    134378
Name: count, dtype: int64

Question length summary:
              q1_len         q2_len
count  363846.000000  363846.000000
mean       10.942822      11.183468
std         5.437638       6.325961
min         1.000000       1.000000
25%         7.000000       7.000000
50%        10.000000      10.000000
75%        13.000000      13.000000
max       125.000000     237.000000


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

def pair_to_text(df):
    return (df["question1"].fillna("") + " [SEP] " + df["question2"].fillna("")).tolist()


BASELINE_TRAIN_SIZE = 50000
baseline_train = train_df.sample(n=min(BASELINE_TRAIN_SIZE, len(train_df)), random_state=SEED)

X_train_base = pair_to_text(baseline_train)
y_train_base = baseline_train["label"].values
X_val_base = pair_to_text(val_df)
y_val = val_df["label"].values

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=100000, ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

baseline.fit(X_train_base, y_train_base)
baseline_preds = baseline.predict(X_val_base)

print("TF-IDF baseline results:")
print(classification_report(y_val, baseline_preds, digits=4))
print("Confusion matrix:", confusion_matrix(y_val, baseline_preds))

TF-IDF baseline results:
              precision    recall  f1-score   support

           0     0.8058    0.7932    0.7994     25545
           1     0.6544    0.6720    0.6630     14885

    accuracy                         0.7486     40430
   macro avg     0.7301    0.7326    0.7312     40430
weighted avg     0.7501    0.7486    0.7492     40430

Confusion matrix: [[20262  5283]
 [ 4883 10002]]


In [9]:
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["question1"],
        batch["question2"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["question1", "question2", "idx"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch")

print(tokenized_dataset)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 390965
    })
})


In [10]:
TRAIN_SIZE = 30000
EVAL_SIZE = 5000

train_data = tokenized_dataset["train"].shuffle(seed=SEED).select(range(min(TRAIN_SIZE, len(tokenized_dataset["train"]))))
eval_data = tokenized_dataset["validation"].shuffle(seed=SEED).select(range(min(EVAL_SIZE, len(tokenized_dataset["validation"]))))

print(train_data)
print(eval_data)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 30000
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


In [11]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

training_args = TrainingArguments(
    output_dir="./bert-qqp-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.06,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=eval_data,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will b

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.371936,0.335967,0.856600,0.776534,0.860766,0.816483
2,0.242451,0.362201,0.862000,0.790605,0.853751,0.820965


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.371936,0.335967,0.856600,0.776534,0.860766,0.816483
2,0.242451,0.362201,0.862000,0.790605,0.853751,0.820965
3,0.169923,0.533796,0.858200,0.799163,0.824609,0.811687


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=5625, training_loss=0.272743015882704, metrics={'train_runtime': 2306.0068, 'train_samples_per_second': 39.029, 'train_steps_per_second': 2.439, 'total_flos': 5919998745600000.0, 'train_loss': 0.272743015882704, 'epoch': 3.0})

In [12]:

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
import numpy as np
import torch
import gc

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def run_experiment(
    model_name,
    max_length=128,
    learning_rate=2e-5,
    train_size=30000,
    eval_size=5000,
    epochs=3,
    batch_size=16
):
    print("=" * 80)
    print(f"Model: {model_name}")
    print(f"MAX_LENGTH: {max_length}")
    print(f"Learning rate: {learning_rate}")
    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_function(example):
        return tokenizer(
            example["question1"],
            example["question2"],
            truncation=True,
            max_length=max_length
        )

    tokenized = dataset.map(tokenize_function, batched=True)

    train_data = tokenized["train"].shuffle(seed=SEED).select(
        range(min(train_size, len(tokenized["train"])))
    )

    eval_data = tokenized["validation"].shuffle(seed=SEED).select(
        range(min(eval_size, len(tokenized["validation"])))
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    output_dir = f"./results_{model_name.replace('/', '_')}_len{max_length}_lr{learning_rate}"

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=100,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
        data_collator=data_collator
    )

    trainer.train()
    results = trainer.evaluate()

    # free GPU memory after experiment
    del model
    del trainer
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "model": model_name,
        "max_length": max_length,
        "learning_rate": learning_rate,
        "train_size": train_size,
        "eval_size": eval_size,
        "epochs": epochs,
        "accuracy": results["eval_accuracy"],
        "precision": results["eval_precision"],
        "recall": results["eval_recall"],
        "f1": results["eval_f1"],
    }

In [13]:
experiment_results = []

experiments = [
    {
        "model_name": "bert-base-uncased",
        "max_length": 128,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "distilbert-base-uncased",
        "max_length": 128,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "roberta-base",
        "max_length": 128,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 64,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 256,
        "learning_rate": 2e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 128,
        "learning_rate": 1e-5,
    },
    {
        "model_name": "bert-base-uncased",
        "max_length": 128,
        "learning_rate": 3e-5,
    },
]

for exp in experiments:
    result = run_experiment(
        model_name=exp["model_name"],
        max_length=exp["max_length"],
        learning_rate=exp["learning_rate"],
        train_size=30000,
        eval_size=5000,
        epochs=3,
        batch_size=16
    )
    experiment_results.append(result)

results_df = pd.DataFrame(experiment_results)
results_df = results_df.sort_values(by="f1", ascending=False)

display(results_df)

Model: bert-base-uncased
MAX_LENGTH: 128
Learning rate: 2e-05


Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [14]:
bert_metrics = trainer.evaluate()
print(bert_metrics)

{'eval_loss': 0.36324062943458557, 'eval_accuracy': 0.862, 'eval_precision': 0.7906046976511744, 'eval_recall': 0.8537506745817594, 'eval_f1': 0.820965230928905, 'eval_runtime': 42.1617, 'eval_samples_per_second': 118.591, 'eval_steps_per_second': 3.724, 'epoch': 3.0}


In [15]:
pred_output = trainer.predict(eval_data)
logits = pred_output.predictions
labels = pred_output.label_ids
preds = np.argmax(logits, axis=-1)

print(classification_report(labels, preds, target_names=["not_paraphrase", "paraphrase"], digits=4))
print("Confusion matrix:", confusion_matrix(labels, preds))

                precision    recall  f1-score   support

not_paraphrase     0.9096    0.8669    0.8877      3147
    paraphrase     0.7906    0.8538    0.8210      1853

      accuracy                         0.8620      5000
     macro avg     0.8501    0.8603    0.8543      5000
  weighted avg     0.8655    0.8620    0.8630      5000

Confusion matrix: [[2728  419]
 [ 271 1582]]


In [16]:
from scipy.special import softmax

probs = softmax(logits, axis=1)[:, 1]
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for threshold in thresholds:
    tuned_preds = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, tuned_preds, average="binary", zero_division=0)
    rows.append({"threshold": threshold, "precision": precision, "recall": recall, "f1": f1})

threshold_df = pd.DataFrame(rows)
display(threshold_df.sort_values("f1", ascending=False).head(10))

best_threshold = threshold_df.sort_values("f1", ascending=False).iloc[0]["threshold"]
print("Best threshold:", best_threshold)

,threshold,precision,recall,f1
7,0.45,0.782609,0.864544,0.821538
8,0.50,0.790605,0.853751,0.820965
9,0.55,0.797352,0.845116,0.820540
6,0.40,0.774364,0.870480,0.819614
10,0.60,0.803952,0.834323,0.818856
4,0.30,0.757953,0.887210,0.817504
5,0.35,0.765176,0.877496,0.817496
11,0.65,0.811532,0.820291,0.815888
3,0.25,0.744719,0.894226,0.812653
2,0.20,0.736516,0.899083,0.809721


Best threshold: 0.45000000000000007


In [17]:

eval_size = len(eval_data)
raw_eval_sample = dataset["validation"].shuffle(seed=SEED).select(range(min(EVAL_SIZE, len(dataset["validation"]))))
errors_df = pd.DataFrame(raw_eval_sample)
errors_df["pred"] = preds
errors_df["paraphrase_probability"] = probs
errors_df["error_type"] = np.where(
    (errors_df["label"] == 0) & (errors_df["pred"] == 1), "false_positive",
    np.where((errors_df["label"] == 1) & (errors_df["pred"] == 0), "false_negative", "correct")
)

print("False positives:")
display(errors_df[errors_df["error_type"] == "false_positive"][["question1", "question2", "label", "pred", "paraphrase_probability"]].head(10))

print("False negatives:")
display(errors_df[errors_df["error_type"] == "false_negative"][["question1", "question2", "label", "pred", "paraphrase_probability"]].head(10))

False positives:


,question1,question2,label,pred,paraphrase_probability
0,How do I get rid of anxiety and depression?,"How do I get rid of anxiety, loneliness and de...",0,1,0.858387
21,How do I make dreams come true successfully?,How can I make my dreams come true?,0,1,0.895023
24,Can Rahul Gandhi change the current plight of ...,Massive response for Rahul Gandhi in UP khat y...,0,1,0.609363
38,Where is the strangest place you've ever mastu...,Where is the weirdest place you've had sex?,0,1,0.928825
52,What are the best new Car technology that most...,What are the best new car products or inventio...,0,1,0.920662
59,How did you learn data mining?,How do I learn data mining algorithms?,0,1,0.527104
62,How much money can I withdraw through self che...,How much money can I withdraw through cheque?,0,1,0.660402
64,What type of guys do girls like?,What type of girls do guys like?,0,1,0.599405
68,How often does Google Maps update their satell...,Does the Google Map in satellite mode ever get...,0,1,0.703131
76,What are Aam Aadmi party's chances in the Goa ...,How many seats will AAP be able to secure if i...,0,1,0.753109


False negatives:


,question1,question2,label,pred,paraphrase_probability
36,What are the pros and cons of buying a used ca...,Is buying used cars from Avis or Hertz a good ...,1,0,0.344564
53,What do you think of the new iPhone7 Airpods?,What do you think of Apple's airpods?,1,0,0.072938
107,What are the best places to visit near Bangalo...,What are some good places to visit near Bengal...,1,0,0.006461
117,What is the point of living when were all goin...,What is the point of living if we are going to...,1,0,0.084348
118,Does the Milky Way Galaxy have an orbit? What ...,Does the Milky Way revolve around anything?,1,0,0.499062
143,Why do people eat cat meat?,Why do people eat cat?,1,0,0.338204
148,Are/were mermaids/mermen real?,Are maremaids real?,1,0,0.002599
164,How should I make a good film?,How do I make an indie film?,1,0,0.356988
204,Is Hillary Clinton's enabling of voter fraud a...,What is your opinion on the O'Keefe video expo...,1,0,0.003113
229,How much does a sheet of paper weigh?,How much does 8 sheets of paper weigh?,1,0,0.029453
